# GeneRA: Rheumatoid Arthritis Prediction

This beginner-friendly notebook follows the same workflow as the original project. It uses the same GEO datasets, sample-selection rules, labels, 300-feature selection, 80:20 train/test split, MLP architecture, evaluation metrics, plots, and saved files.

**Important:** The code preserves the original processing choices so results remain comparable. Exact scores can still vary slightly because neural-network training can be stochastic.

## 1. Install libraries

These packages download GEO datasets, process data, train the neural network, and draw graphs.

In [ ]:
%pip -q install GEOparse tensorflow scikit-learn pandas numpy matplotlib seaborn joblib

## 2. Import libraries and set the random seed

The imports provide data handling, preprocessing, model training, evaluation, and saving. The seed makes runs more reproducible.

In [ ]:
import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse
import tensorflow as tf
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

OUTPUT_DIR = Path("/content/GeneRA_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Output folder:", OUTPUT_DIR)

## 3. Download the datasets

GSE97810 contains the rheumatoid arthritis cohort. GSE97475 provides the healthy-control cohort.

In [ ]:
# GSE97810: RA-MAP baseline RA cohort
# GSE97475: healthy hepatitis-B vaccine-recipient cohort
# Both series use GPL10904, Illumina HumanHT-12 V4.0.

GEO_DIR = Path("/content/geo_data")
GEO_DIR.mkdir(parents=True, exist_ok=True)

gse_ra = GEOparse.get_GEO("GSE97810", destdir=str(GEO_DIR), annotate_gpl=False)
gse_control = GEOparse.get_GEO("GSE97475", destdir=str(GEO_DIR), annotate_gpl=False)

print("GSE97810 RA samples:", len(gse_ra.gsms))
print("GSE97475 healthy-control samples:", len(gse_control.gsms))
print("RA platforms:", list(gse_ra.gpls.keys()))
print("Control platforms:", list(gse_control.gpls.keys()))

if not gse_ra.gsms or not gse_control.gsms:
    raise RuntimeError("GEO download completed without sample records. Re-run this cell after checking the Colab internet connection.")

## 4. Read sample information

GEO metadata describes each sample. These functions turn the annotations into a table that can be searched.

In [ ]:
def flatten_value(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return " | ".join(str(item) for item in value)
    return str(value)

def make_metadata(gse, cohort_name):
    rows = []
    for gsm_id, sample in gse.gsms.items():
        row = {"sample_id": str(gsm_id), "cohort": cohort_name}
        for key, value in sample.metadata.items():
            key = str(key).lower()
            row[key] = flatten_value(value)

            if key.startswith("characteristics"):
                values = value if isinstance(value, (list, tuple, np.ndarray)) else [value]
                for item in values:
                    item = str(item)
                    if ":" in item:
                        field_name, field_value = item.split(":", 1)
                        field_name = re.sub(r"[^a-z0-9]+", "_", field_name.strip().lower()).strip("_")
                        row[f"characteristic_{field_name}"] = field_value.strip()
        rows.append(row)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).set_index("sample_id").fillna("")

meta_ra = make_metadata(gse_ra, "RA")
meta_control = make_metadata(gse_control, "Control")

print("RA metadata shape:", meta_ra.shape)
print("Control metadata shape:", meta_control.shape)

def combined_annotation(row):
    return " | ".join(f"{col}: {row[col]}" for col in row.index if str(row[col]).strip())

meta_ra["annotation_text"] = meta_ra.apply(combined_annotation, axis=1).str.lower()
meta_control["annotation_text"] = meta_control.apply(combined_annotation, axis=1).str.lower()

# Show key GEO sample annotations.
def useful_columns(df):
    keys = ("title", "source_name", "characteristics", "tissue", "cell type", "patient", "visit", "time")
    return [c for c in df.columns if any(k in c.lower() for k in keys)]

display(meta_ra[useful_columns(meta_ra)].head(8))
display(meta_control[useful_columns(meta_control)].head(8))

## 5. Select comparable samples

Keep baseline RA samples and healthy control samples. Then select a tissue/cell type found in both datasets so the comparison is more consistent.

In [ ]:
def get_source_text(row):
    parts = []
    for col in row.index:
        if any(token in col.lower() for token in ("source_name", "title", "description")):
            parts.append(str(row[col]))
    return " | ".join(parts).lower()

def get_tissue_text(row):
    parts = []
    for col in row.index:
        normalized_col = str(col).lower()
        if (
            "tissue" in normalized_col
            or "cell type" in normalized_col
            or "cell_type" in normalized_col
        ):
            parts.append(str(row[col]))
    return " | ".join(parts).lower()

def canonical_tissue(value):
    value = str(value).lower().replace("_", " ").replace("-", " ")
    value = re.sub(r"\s+", " ", value).strip()
    if "whole blood" in value:
        return "whole_blood"
    if "cd14" in value:
        return "cd14"
    if "cd4" in value:
        return "cd4"
    if "cd8" in value:
        return "cd8"
    if "pbmc" in value:
        return "pbmc"
    return None

def prepare_candidates(meta, cohort):
    out = meta.copy()
    out["source_text"] = out.apply(get_source_text, axis=1)
    out["tissue_text"] = out.apply(get_tissue_text, axis=1)
    out["tissue_group"] = out["tissue_text"].map(canonical_tissue)

    if cohort == "RA":
        baseline = out["source_text"].str.contains(r"\bbaseline\b", regex=True, na=False)
        out = out[baseline].copy()
    else:
        healthy = out["source_text"].str.contains(r"\bhealthy\b", regex=True, na=False)
        out = out[healthy].copy()

    return out[out["tissue_group"].notna()].copy()

ra_candidates = prepare_candidates(meta_ra, "RA")
control_candidates = prepare_candidates(meta_control, "Control")

print("RA baseline candidates by tissue:")
display(ra_candidates["tissue_group"].value_counts().rename("samples").to_frame())
print("Healthy candidates by tissue:")
display(control_candidates["tissue_group"].value_counts().rename("samples").to_frame())

shared_tissues = set(ra_candidates["tissue_group"]) & set(control_candidates["tissue_group"])
preference = ["whole_blood", "pbmc", "cd14", "cd4", "cd8"]
selected_tissue = next((t for t in preference if t in shared_tissues), None)

if selected_tissue is None:
    raise RuntimeError(
        "No shared tissue/cell type could be verified between GSE97810 and GSE97475. "
        "Review the displayed GEO annotation tables; do not combine different specimen types."
    )

ra_selected_meta = ra_candidates[ra_candidates["tissue_group"] == selected_tissue].copy()
control_selected_meta = control_candidates[control_candidates["tissue_group"] == selected_tissue].copy()

print("Selected comparable tissue:", selected_tissue)
print("RA candidates:", len(ra_selected_meta))
print("Control candidates:", len(control_selected_meta))

## 6. Remove repeated participants and create labels

Keep one sample per participant when IDs are available. Label RA as 1 and Control as 0.

In [ ]:
def participant_id(row):
    for col in row.index:
        key = re.sub(r"[^a-z0-9]", "", str(col).lower())
        if (
            key in {"patientid", "participantid", "subjectid", "donorid"}
            or key.endswith("patientid")
            or key.endswith("participantid")
            or key.endswith("subjectid")
            or key.endswith("donorid")
        ):
            value = str(row[col]).strip()
            if value and value.lower() not in {"nan", "na", "none", "null"}:
                return value
    return None

def deduplicate_participants(df):
    df = df.copy()
    df["participant_id"] = df.apply(participant_id, axis=1)
    has_ids = df["participant_id"].notna().sum()
    if has_ids == len(df) and len(df):
        df = df.drop_duplicates(subset=["participant_id"], keep="first")
        return df, True
    return df.drop(columns=["participant_id"]), False

ra_selected_meta, ra_ids_found = deduplicate_participants(ra_selected_meta)
control_selected_meta, control_ids_found = deduplicate_participants(control_selected_meta)

if not (ra_ids_found and control_ids_found):
    warnings.warn(
        "A complete participant ID was not available for every selected sample in both cohorts. "
        "A participant-independent split cannot be guaranteed."
    )

if min(len(ra_selected_meta), len(control_selected_meta)) < 5:
    raise RuntimeError(
        "Fewer than five samples remain in at least one class after tissue and participant checks. "
        "The data are insufficient for a reliable train/test experiment."
    )

ra_ids = ra_selected_meta.index.astype(str).tolist()
control_ids = control_selected_meta.index.astype(str).tolist()
selected_ids = ra_ids + control_ids

y = pd.Series(
    [1] * len(ra_ids) + [0] * len(control_ids),
    index=selected_ids,
    name="label",
    dtype="int32"
)

print("Final selected samples:", len(selected_ids))
print("RA samples:", int((y == 1).sum()))
print("Control samples:", int((y == 0).sum()))
print("Tissue group:", selected_tissue)

## 7. Build the gene-expression table

Each row becomes a sample and each column becomes a gene/probe measurement. Missing values are filled using the gene median.

In [ ]:
def expression_series(sample, gsm_id):
    table = sample.table.copy()
    if table.empty:
        raise RuntimeError(f"GEO sample {gsm_id} has an empty expression table.")

    id_col = "ID_REF" if "ID_REF" in table.columns else table.columns[0]
    value_col = "VALUE" if "VALUE" in table.columns else None
    if value_col is None:
        numeric_cols = [c for c in table.columns if c != id_col and pd.to_numeric(table[c], errors="coerce").notna().any()]
        if not numeric_cols:
            raise RuntimeError(f"No numeric expression column was found for {gsm_id}.")
        value_col = numeric_cols[0]

    temp = table[[id_col, value_col]].copy()
    temp[id_col] = temp[id_col].astype(str)
    temp[value_col] = pd.to_numeric(temp[value_col], errors="coerce")
    temp = temp.replace([np.inf, -np.inf], np.nan).dropna()
    temp = temp.drop_duplicates(subset=[id_col], keep="first")
    return temp.set_index(id_col)[value_col].rename(str(gsm_id))

sample_series = {}
for gsm_id in ra_ids:
    sample_series[gsm_id] = expression_series(gse_ra.gsms[gsm_id], gsm_id)
for gsm_id in control_ids:
    sample_series[gsm_id] = expression_series(gse_control.gsms[gsm_id], gsm_id)

expression = pd.concat(sample_series.values(), axis=1, join="inner")
expression = expression.loc[:, selected_ids]
expression = expression.T

# Remove unusable gene measurements.
expression = expression.apply(pd.to_numeric, errors="coerce")
expression = expression.replace([np.inf, -np.inf], np.nan)
expression = expression.dropna(axis=1, how="all")
expression = expression.loc[:, expression.nunique(dropna=True) > 1]

# Fill missing values with each gene's median.
expression = expression.fillna(expression.median(axis=0))
expression = expression.dropna(axis=1, how="any")

if expression.empty or expression.shape[1] < 2:
    raise RuntimeError("The matched expression matrix is empty or has too few usable features.")

# Keep sample and label order aligned.
X = expression.loc[selected_ids].copy()
y = y.loc[X.index]

print("Expression matrix (samples x genes/probes):", X.shape)
print("Missing values remaining:", int(X.isna().sum().sum()))
display(X.iloc[:5, :5])

## 8. Split training and testing data

Use 80% of samples to train the model and 20% to test it. Stratification keeps the RA/control proportions similar.

In [ ]:
# Split samples into training and testing sets; keep class proportions similar.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise RuntimeError("The split does not contain both classes in train and test.")

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training class counts:")
display(y_train.map({0: "Control", 1: "RA"}).value_counts())
print("Testing class counts:")
display(y_test.map({0: "Control", 1: "RA"}).value_counts())

## 9. Plot the class distribution

This graph shows how many RA and control samples were selected.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x=y.map({0: "Control", 1: "RA"}), order=["Control", "RA"])
plt.title(f"GeneRA sample distribution — {selected_tissue}")
plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.show()

## 10. Select 300 features and scale them

SelectKBest with the ANOVA F-test ranks genes by how strongly their values differ between the two classes. StandardScaler puts selected values on a comparable scale.

In [ ]:
k = min(300, X_train.shape[1])
selector = SelectKBest(score_func=f_classif, k=k)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected).astype(np.float32)
X_test_scaled = scaler.transform(X_test_selected).astype(np.float32)

if not np.isfinite(X_train_scaled).all() or not np.isfinite(X_test_scaled).all():
    raise RuntimeError("Non-finite values remain after preprocessing.")

print("Selected features:", k)
print("Training array:", X_train_scaled.shape)
print("Testing array:", X_test_scaled.shape)

## 11. Build and train the MLP

The network has 300 input values, hidden layers with 128 and 64 neurons, dropout, and one sigmoid output. Early stopping monitors validation loss.

In [ ]:
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation="relu"),
    Dropout(0.30),
    Dense(64, activation="relu"),
    Dropout(0.20),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

# Keep some training data for validation during learning.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_scaled,
    y_train.to_numpy(),
    test_size=0.20,
    random_state=SEED,
    stratify=y_train.to_numpy()
)

history = model.fit(
    X_fit,
    y_fit,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=min(16, max(4, len(X_fit) // 4)),
    callbacks=[early_stop],
    verbose=1
)

model.summary()

## 12. Plot training history

The graphs show training and validation accuracy and loss across epochs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Model Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Model Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## 13. Evaluate the test data

The model predicts RA/control labels. Accuracy, precision, recall, F1-score, ROC-AUC, and the classification report summarize performance.

In [ ]:
y_scores = model.predict(X_test_scaled, verbose=0).ravel()
y_pred = (y_scores >= 0.5).astype(int)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_scores) if y_test.nunique() == 2 else np.nan
}

print("GeneRA MLP test metrics")
for name, value in metrics.items():
    print(f"{name:10s}: {value:.4f}" if np.isfinite(value) else f"{name:10s}: not available")

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Control", "RA"],
    zero_division=0
))

## 14. Confusion matrix and ROC curve

The confusion matrix counts correct and incorrect predictions. The ROC curve shows the trade-off between true-positive and false-positive rates.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Control", "RA"],
    yticklabels=["Control", "RA"]
)
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("GeneRA Confusion Matrix")
plt.tight_layout()
plt.show()

if y_test.nunique() == 2:
    RocCurveDisplay.from_predictions(y_test, y_scores)
    plt.title("GeneRA ROC Curve")
    plt.tight_layout()
    plt.show()

## 15. Compare with Logistic Regression

This traditional machine-learning model is a baseline for comparison with the MLP.

In [ ]:
baseline = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=SEED
)
baseline.fit(X_train_scaled, y_train)
baseline_pred = baseline.predict(X_test_scaled)

print("Logistic Regression accuracy:", round(accuracy_score(y_test, baseline_pred), 4))
print("MLP accuracy:", round(accuracy_score(y_test, y_pred), 4))

## 16. Save the results

Save the trained model, feature selector, scaler, feature names, selected sample list, and test metrics.

In [ ]:
model_path = OUTPUT_DIR / "GeneRA_MLP_Model.keras"
selector_path = OUTPUT_DIR / "GeneRA_Feature_Selector.pkl"
scaler_path = OUTPUT_DIR / "GeneRA_Scaler.pkl"
feature_path = OUTPUT_DIR / "GeneRA_Input_Feature_Names.pkl"
samples_path = OUTPUT_DIR / "GeneRA_Selected_Samples.csv"
metrics_path = OUTPUT_DIR / "GeneRA_Test_Metrics.json"

model.save(model_path)
joblib.dump(selector, selector_path)
joblib.dump(scaler, scaler_path)
joblib.dump(list(X.columns), feature_path)

sample_report = pd.DataFrame({
    "sample_id": selected_ids,
    "cohort": ["RA"] * len(ra_ids) + ["Control"] * len(control_ids),
    "label": [1] * len(ra_ids) + [0] * len(control_ids),
    "tissue_group": selected_tissue
})
sample_report.to_csv(samples_path, index=False)

with open(metrics_path, "w") as f:
    json.dump({k: (float(v) if np.isfinite(v) else None) for k, v in metrics.items()}, f, indent=2)

print("Saved files:")
for path in [model_path, selector_path, scaler_path, feature_path, samples_path, metrics_path]:
    print("-", path)